<a href="https://colab.research.google.com/github/drizzy765/flyrank_internship_ML-01_assignment/blob/main/w02_ml_task_framing_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2: Frame Your Lane as an ML Task

This notebook bridges the gap between the business problem (identifying underperforming content) and the concrete Machine Learning architecture needed to solve it.

## 1) My lane as an ML task (type)

**Task Type: Regression driving a Scoring/Ranking System**

To identify CTR anomalies, we must first establish a baseline expectation. This requires a **Regression** model to predict what a "normal" CTR should be for a specific page based on its features (position, age, query intent). Once we predict that baseline, the final output becomes a **Ranking/Scoring** task, where we score pages based on the magnitude of their negative residual (Expected CTR vs. Actual CTR).

## 2) Target or proxy

**The Target Variable:** `ctr` (Click-Through Rate). The regression model is trained to predict this continuous value.

**The Actionable Proxy:** `ctr_opportunity_gap`. This is a derived score calculated as `Predicted Expected CTR - Actual CTR`. A high positive gap indicates a severe anomaly (the page is getting far fewer clicks than mathematically expected).

## 3) Success metric

There are two layers of success metrics for this pipeline:

1. **Technical Metric (Model Level):** `MAE` (Mean Absolute Error) or `RMSE`. We want our regression model's baseline predictions to be as accurate as possible across the dataset.
2. **Business Metric (Action Level):** `Precision@50`. Because this model generates a ranked queue for human review, accuracy across all 500,000 pages doesn't matter as much as the top of the list. If an SEO Specialist reviews the top 50 flagged URLs, what percentage of them are genuine, actionable anomalies rather than data noise?

## 4) The unit of analysis, as a real dataframe

Below, we load the starter dataset to explicitly show the grain of our data.
**One Row = One pseudonymized content item (`content_id`) aggregated over a 90-day window.**

In [ ]:
from google.colab import files
files.upload()

In [2]:
import pandas as pd
import os


In [3]:

data_path = 'content_refresh_anonymized.csv'

In [7]:


df = pd.read_csv(data_path)
print(" 1. The Unit of Analysis")
print("One row represents one content item's performance aggregated over 90 days.\n")

# Isolate the features relevant to our CTR task
unit_df = df[['content_id', 'avg_position', 'impressions_90d', 'clicks_90d', 'ctr', 'content_age_days']].head(3)
display(unit_df)

print(" 2. Sketching the Target & Proxy")
print("To build our scoring system, the ML model predicts 'expected_ctr'.")
print("We then calculate the 'opportunity_score_gap' to rank pages.\n")





 1. The Unit of Analysis
One row represents one content item's performance aggregated over 90 days.



,content_id,avg_position,impressions_90d,clicks_90d,ctr,content_age_days
0,content_304f48230142,10.6,3803,29,0.76,187
1,content_a1fb4e703a9e,20.3,15320,7,0.05,445
2,content_9aa793d4d895,36.5,12581,11,0.09,141


 2. Sketching the Target & Proxy
To build our scoring system, the ML model predicts 'expected_ctr'.
We then calculate the 'opportunity_score_gap' to rank pages.



In [5]:
# Simulating what the output of the regression model will look like:
# (Mocking predicted values to show the dataframe structure)
unit_df['expected_ctr_prediction'] = [0.045, 0.220, 0.015]
unit_df['opportunity_score_gap'] = unit_df['expected_ctr_prediction'] - unit_df['ctr']

display(unit_df[['content_id', 'avg_position', 'ctr', 'expected_ctr_prediction', 'opportunity_score_gap']])

,content_id,avg_position,ctr,expected_ctr_prediction,opportunity_score_gap
0,content_304f48230142,10.6,0.76,0.045,-0.715
1,content_a1fb4e703a9e,20.3,0.05,0.220,0.170
2,content_9aa793d4d895,36.5,0.09,0.015,-0.075


## 5) Why ML beats a fixed rule here

A fixed SQL rule (e.g., `WHERE avg_position < 10 AND ctr < 0.02`) is rigid and brittle. It assumes CTR degrades linearly across positions and treats all content types equally.

In reality, CTR is a highly non-linear, multivariate function. A #3 position for an informational query might naturally have a 5% CTR, while a #3 position for a transactional query might demand a 15% CTR. Machine Learning (like XGBoost or Random Forest) easily captures these complex, non-linear feature interactions (position × intent × age) to create a dynamic, accurate baseline that a simple `GROUP BY` average could never achieve.